# IMPORTANDO BIBLIOTECAS E DADOS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import math

In [2]:
data = pd.read_csv("D:\Camilla_\SIAC\data\input\df_total.csv")

In [3]:
data.head()

,date-time,364602_31231_FI_303,364602_31231_FI_305,364602_31231_PI_302,364602_31231_PI_306,364602_31231_PDI_303,364602_31231_PDI_304,364602_31231_TI_303,364602_31231_TI_314,364602_31231_TI_315,364602_31231_TI_316,364602_31231_TI_317,364602_31231_TI_318,364602_31231_TI_324,364602_31231_TI_309
0,2022-01-01 11:00:00,-1704.699463,0.397332,2.071551,1.647949,-0.013636,-0.035243,32.063946,26.564144,27.225803,27.160862,27.342075,27.433908,27.059429,28.062088
1,2022-01-01 11:10:00,-1627.635864,0.400234,2.076229,1.556396,-0.029954,-0.043380,31.633373,26.615526,27.273422,27.262171,27.444054,27.485897,27.088348,28.113070
2,2022-01-01 11:20:00,-1611.280518,0.396393,1.834457,1.556396,-0.015143,-0.036058,30.856258,26.584444,27.250973,27.269323,27.477003,27.494776,27.035734,28.116104
3,2022-01-01 11:30:00,-1630.498169,0.394055,1.923238,1.556396,-0.012021,-0.034269,30.497635,26.564489,27.214085,27.285011,27.503239,27.499760,27.009365,28.021307
4,2022-01-01 11:40:00,-1668.774292,0.393517,1.972165,1.556396,-0.011995,-0.033511,30.334396,26.526068,27.181429,27.286526,27.504520,27.509985,26.968203,27.909653


# REGRESSÃO LINEAR 

In [4]:
# Regressão linear: y = A0 + A1x
def reglin(data, alvo):
   
   """ 'data' é o dataset de entrada com as variáveis explicativas.
       'alvo' é a variável alvo, a que vc quer preditar.
   """
    
   # y = variável alvo
   y = data[alvo].to_numpy(dtype=float)
    
   # X = todas as outras variáveis
   x = data.drop(columns=[alvo]).select_dtypes(include='number').to_numpy(dtype=float)
    
   # adiciona uma coluna de 1 para o coeficiente linear
   X = np.column_stack((np.ones(len(x)), x))
   
   
    
   # cálculo dos coeficientes
   coef = np.linalg.lstsq(X, y, rcond=None)[0]
    
   A = coef[0]
   B = coef[1:]
    
   
   y_pred = X @ coef          # valores previstos
    
   # cálculo do R²
   r2 = 1 - (
        np.sum((y - y_pred)**2) /
        np.sum((y - np.mean(y))**2)
   )
   return {
      "modelo": "linear",
      "coef_linear": float(A), 
      "coef_angular": B.tolist(),
      "r2": float(r2)

   }

   

In [5]:
reglin(data,'364602_31231_FI_305')

{'modelo': 'linear',
 'coef_linear': 0.9122585376786316,
 'coef_angular': [1.6208044310955924e-05,
  -0.0002361317359966022,
  -5.607377655174015e-05,
  -0.0030787928155052904,
  -0.0025869695817411684,
  -0.004526359949609788,
  -0.17654153690531782,
  0.16707753872078976,
  -0.25811613765095454,
  0.239510026651314,
  -0.0165031420876267,
  -0.027469950690385506,
  0.052581224644183536],
 'r2': 0.8554916199207433}

# REPRESSÃO PONDERADA

In [6]:
# Regressão Ponderada: pesos variando de 0 a 1, sendo 1 o ponto mais recente
def regpond(data, alvo):

   """ 'data' é o dataset de entrada com as variáveis explicativas.
       'alvo' é a variável alvo, a que vc quer preditar.
   """
   
   # y = variável alvo
   y = data[alvo].to_numpy(dtype=float)
   
   # x = todas as outras variáveis
   x = data.drop(columns=[alvo]).select_dtypes(include='number').to_numpy(dtype=float)

   # adiciona uma coluna de 1 para o coeficiente linear
   X = np.column_stack((np.ones(len(x)), x))
   
   
   n = len(y)
   
   # peso
   w = np.linspace(0, 1, n)      # ou uma ponderação exponencial..? w = np.linspace(1, n, n)
   W = np.diag(w)                # matriz diagonal dos pesos



   # cálculo dos coeficientes da regressão ponderada
   coef = np.linalg.lstsq(W @ X, W @ y, rcond=None)[0]
   A = coef[0]
   B = coef[1:]
   
   
   y_pred = X @ coef         # valores previstos
    
    
       
   # cálculo do R²
   r2 = 1 - (
      np.sum((y - y_pred)**2) /
      np.sum((y - np.mean(y))**2))
   
   
   
   return {
         "modelo": "ponderada",
         "coef_linear": float(A), 
         "coef_angular": B.tolist(),
         "r2": float(r2)
         }
   
      

In [7]:
regpond(data,'364602_31231_FI_305')

{'modelo': 'ponderada',
 'coef_linear': 1.2651075227557786,
 'coef_angular': [1.6835765236114042e-05,
  -0.0010126902738320704,
  0.0007303311410462227,
  0.0006858090573421277,
  -0.0026985739661561704,
  -0.025172286237738213,
  -0.17937268971675893,
  0.1707581613324864,
  -0.3155191821254712,
  0.31399257304092626,
  -0.04660530914827113,
  -0.035462143565088894,
  0.07520952624374573],
 'r2': 0.8444534245150259}

# AUTOREGRESSÃO

In [8]:
def autoregressao(data, data_treino, alvo, var_explicativas, lag_max, n_predicoes, n_treinos):

    """
    data_treino = quantidade de pontos utilizados no primeiro treino.

    Exemplo:
        data_treino = 4320

    O primeiro treino utiliza os pontos:
        0 até 4319

    O segundo treino:
        1 até 4320

    E assim por diante, formando uma janela móvel.
    """

    data = data.copy()
    data['date-time'] = pd.to_datetime(data['date-time'])
    data = data.sort_values('date-time').reset_index(drop=True)

    # Garantir que data_treino seja uma quantidade inteira de pontos
    tamanho_treino = int(data_treino)

    # Verificação
    if tamanho_treino <= 0:
        raise ValueError("data_treino deve ser maior que zero.")

    if tamanho_treino >= len(data):
        raise ValueError(
            f"data_treino ({tamanho_treino}) deve ser menor "
            f"que o número total de pontos do data ({len(data)})."
        )

    # numpy
    X_all = data[var_explicativas].values
    y_all = data[alvo].values
    datas_all = data['date-time'].values

    previsoes = []
    coeficientes = []

    # loop principal
    for treino in range(n_treinos):

        inicio = treino
        fim = inicio + tamanho_treino

        # Não existe quantidade suficiente de dados
        if fim >= len(data):
            break

        idx_base = fim - 1

        # Dados utilizados neste treino
        X_treino = X_all[inicio:fim]
        y_treino = y_all[inicio:fim]

        # loop das lags
        for lag_usado in range(1, lag_max + 1):

            n_amostras = len(y_treino) - lag_usado

            if n_amostras < 10:
                continue

            # montar X e Y
            Y = y_treino[lag_usado:]
            X = X_treino[:-lag_usado]

            # intercepto
            X = np.column_stack((np.ones(len(X)), X))

            # regressão direta
            M = X.T @ X
            b = X.T @ Y

            A = np.linalg.pinv(M) @ b

            coeficientes.append({
                'treino': treino,
                'lag_usado': lag_usado,
                'coeficientes': A
            })

            # previsão
            if lag_usado > n_predicoes:
                continue

            idx_pred = idx_base + lag_usado

            if idx_pred >= len(data):
                continue

            X_pred = np.concatenate(([1], X_all[idx_base])).reshape(1, -1)

            Y_pred = (X_pred @ A)[0]
            Y_real = y_all[idx_pred]

            erro_relativo = (
                abs((Y_real - Y_pred) / Y_real)
                if Y_real != 0 else np.nan
            )

            previsoes.append({
                'treino': treino,
                'lag_usado': lag_usado,
                'data_predicao': datas_all[idx_pred],
                'Y_real': Y_real,
                'Y_pred': Y_pred,
                'erro_relativo': erro_relativo,
                'erro_relativo_percentual': erro_relativo * 100
            })

    resultados = pd.DataFrame(previsoes)
    coef = pd.DataFrame(coeficientes)

    return resultados, coef


In [9]:
data.columns

Index(['date-time', '364602_31231_FI_303', '364602_31231_FI_305',
       '364602_31231_PI_302', '364602_31231_PI_306', '364602_31231_PDI_303',
       '364602_31231_PDI_304', '364602_31231_TI_303', '364602_31231_TI_314',
       '364602_31231_TI_315', '364602_31231_TI_316', '364602_31231_TI_317',
       '364602_31231_TI_318', '364602_31231_TI_324', '364602_31231_TI_309'],
      dtype='str')

In [10]:
var_explicativas = ['364602_31231_FI_303', '364602_31231_PI_302', '364602_31231_PI_306', '364602_31231_PDI_303',
                    '364602_31231_PDI_304', '364602_31231_TI_303', '364602_31231_TI_314', '364602_31231_TI_315',
                    '364602_31231_TI_316', '364602_31231_TI_317', '364602_31231_TI_318', '364602_31231_TI_324',
                    '364602_31231_TI_309', '364602_31231_FI_305']

In [11]:
resultado, coef = autoregressao(data, 4320, '364602_31231_FI_305',var_explicativas, 12, 12, 10)

In [12]:
resultado

,treino,lag_usado,data_predicao,Y_real,Y_pred,erro_relativo,erro_relativo_percentual
0,0,1,2022-01-31 11:00:00,0.397918,0.397768,0.000377,0.037720
1,0,2,2022-01-31 11:10:00,0.398826,0.396871,0.004900,0.490035
2,0,3,2022-01-31 11:20:00,0.398139,0.396443,0.004261,0.426085
3,0,4,2022-01-31 11:30:00,0.396250,0.396367,0.000293,0.029350
4,0,5,2022-01-31 11:40:00,0.393827,0.396571,0.006967,0.696681
...,...,...,...,...,...,...,...
115,9,8,2022-01-31 13:40:00,0.395997,0.396692,0.001755,0.175489
116,9,9,2022-01-31 13:50:00,0.396024,0.397066,0.002631,0.263115
117,9,10,2022-01-31 14:00:00,0.395838,0.397400,0.003947,0.394680
118,9,11,2022-01-31 14:10:00,0.395060,0.397634,0.006515,0.651468


In [13]:
coef

,treino,lag_usado,coeficientes
0,0,1,"[0.07799423852929976, 2.182752917467834e-06, -..."
1,0,2,"[0.16254850814675592, 4.268334331036794e-06, -..."
2,0,3,"[0.22986610698626464, 6.126590575084023e-06, -..."
3,0,4,"[0.28165346873402086, 7.3189123512285015e-06, ..."
4,0,5,"[0.32129641913388696, 8.108208203738698e-06, -..."
...,...,...,...
115,9,8,"[0.3896432421024656, 7.306911867122068e-06, -0..."
116,9,9,"[0.3943409915295888, 6.933546249976441e-06, -0..."
117,9,10,"[0.39111510402153726, 6.129737311344474e-06, -..."
118,9,11,"[0.39427925602149116, 5.110365560834218e-06, -..."


# AUTOREGRESSÃO PONDERADA

In [16]:
def autoregressao_ponderada(data, data_treino, alvo, var_explicativas, lag_max, n_predicoes, n_treinos): 
   
        
    data = data.copy()
    data['date-time'] = pd.to_datetime(data['date-time'])
    data = data.sort_values('date-time').reset_index(drop=True)
    
    
    # garantir que data_treiino seja uma quantidade inteira de pontos
    tamanho_treino = int(data_treino)
    
   
    # verificção 
    if tamanho_treino <=0:
       raise ValueError("data_treino deve ser maior que zero.")
   
    if tamanho_treino >= len(data):
       raise ValueError(f"data_treino ({tamanho_treino}) deve ser menor que o número total de pontos do data({len(data)})")
    
    # converter para numpy
    X_all = data[var_explicativas].values
    y_all = data[alvo].values
    datas_all = data['date-time'].values
    
    
    previsoes = []                              #guarda os resultados da previsão
    coeficientes = []                           #guarda os coeficientes
    
    # loop principal
    for treino in range(n_treinos):
        
        inicio = treino
        fim = inicio +tamanho_treino
        
        if fim >= len(data):
            break

        idx_base = fim - 1
        
       #slicing direto (sem copy)
        X_treino = X_all[inicio:fim]
        y_treino = y_all[inicio:fim] 
        
        
        #loop das lags
        for lag_usado in range(1, lag_max + 1):

            n_amostras = len(y_treino) - lag_usado

            if n_amostras < 10:
                continue

            #construir X e Y sem pandas
            Y = y_treino[lag_usado:]
            X = X_treino[:-lag_usado]

            #adicionar intercepto
            X = np.column_stack((np.ones(len(X)), X))

            #pesos lineares
            pesos = np.linspace(0, 1, len(X))

            #aplicar pesos (SEM MATRIZ DIAGONAL)
            Xw = X * pesos[:, None]
            Yw = Y * pesos

            #regressão
            M = X.T @ Xw
            b = X.T @ Yw

            A = np.linalg.pinv(M) @ b

            coeficientes.append({
                'treino': treino,
                'lag_usado': lag_usado,
                'coeficientes': A
            })

            #previsão
            if lag_usado > n_predicoes:
                continue

            idx_pred = idx_base + lag_usado

            if idx_pred >= len(data):
                continue

            X_pred = np.concatenate(([1], X_all[idx_base])).reshape(1, -1)

            Y_pred = (X_pred @ A)[0]
            Y_real = y_all[idx_pred]

            erro_relativo = (
                abs((Y_real - Y_pred) / Y_real)
                if Y_real != 0 else np.nan
            )

            previsoes.append({
                'treino': treino,
                'lag_usado': lag_usado,
                'peso_pred': pesos[-1],
                'data_predicao': datas_all[idx_pred],
                'Y_real': Y_real,
                'Y_pred': Y_pred,
                'erro_relativo': erro_relativo,
                'erro_relativo_percentual': erro_relativo * 100
            })

    resultados = pd.DataFrame(previsoes)
    coef = pd.DataFrame(coeficientes)



    return resultados, coef


In [17]:
resultado_ponderado, coef_ponderado = autoregressao_ponderada(data, 4320, "364602_31231_FI_305", var_explicativas, 12,12,10)

In [18]:
resultado_ponderado

,treino,lag_usado,peso_pred,data_predicao,Y_real,Y_pred,erro_relativo,erro_relativo_percentual
0,0,1,1.0,2022-01-31 11:00:00,0.397918,0.397889,0.000073,0.007273
1,0,2,1.0,2022-01-31 11:10:00,0.398826,0.396947,0.004711,0.471135
2,0,3,1.0,2022-01-31 11:20:00,0.398139,0.396453,0.004235,0.423470
3,0,4,1.0,2022-01-31 11:30:00,0.396250,0.396311,0.000153,0.015275
4,0,5,1.0,2022-01-31 11:40:00,0.393827,0.396503,0.006794,0.679370
...,...,...,...,...,...,...,...,...
115,9,8,1.0,2022-01-31 13:40:00,0.395997,0.396530,0.001345,0.134491
116,9,9,1.0,2022-01-31 13:50:00,0.396024,0.397007,0.002483,0.248270
117,9,10,1.0,2022-01-31 14:00:00,0.395838,0.397399,0.003944,0.394399
118,9,11,1.0,2022-01-31 14:10:00,0.395060,0.397581,0.006380,0.637999
